- provjeri da li da većan broj JOBOVA na scib
- umjesto dva odvojena csv narpavi da is scib i scraph ide sve u jedan te isti csv file

In [ ]:

%matplotlib inline
%reload_ext autoreload
%autoreload 2

work_folder = '/g/stegle/spiljak/benchmarking-cellina/results'
cellina_reproducibility_folder ='/g/stegle/spiljak/cellina_tutorial/cellina-reproducibility/'

import numpy as np
import scanpy as sc
import anndata as ad
import pandas as pd
import os
import seaborn as sns
import matplotlib.pyplot as plt
import sys

from tqdm import tqdm
from sklearn.model_selection import train_test_split

sys.path.append(f'{cellina_reproducibility_folder}/scripts')
from utils import set_seed
from train_loo import preprocess_adata

sys.path.append(f'{cellina_reproducibility_folder}/notebooks/application')
from helpers import _normalize_counts, safe_log2_fold_change, compute_correlations, subsample_adata

from scgraph import scGraph
import scib_metrics
from scib_metrics.benchmark import Benchmarker, BioConservation, BatchCorrection
import scgraph.scgraph as _sg

# Patrhing scGraph - scGraph expects log transformed data, but I can't log trasnform the adata bcs teh scrgpah loads on it's own adata from the file, so I will patch the process_batches function to log transform the data before processing
_orig = _sg.scGraph.process_batches

_orig = _sg.scGraph.process_batches

def _patched(self):
    sc.pp.normalize_total(self.adata, target_sum=1e4)
    sc.pp.log1p(self.adata)
    _orig(self)

_sg.scGraph.process_batches = _patched


labels_key = '_scvi_labels'
domains_key = '_scvi_domains'
batch_key = 'sid' #_scvi_batch

slide_ids = ['110', '120', '210', '221', '222', '231', '232', '242']


for slide_id in slide_ids:
# scGraph metrics -------------------------------


    # Initialize the graph analyzer
    scgraph = scGraph(
        adata_path=f'{work_folder}/{slide_id}/latents_adata.h5ad',   # Path to AnnData object
        batch_key=batch_key,                     # Column name for batch information
        label_key=labels_key,                 # Column name for cell type labels
        trim_rate=0.05,                        # Trim rate for robust mean calculation
        thres_batch=100,                       # Minimum number of cells per batch
        thres_celltype=10,                    # Minimum number of cells per cell type
        only_umap=True,                        # Only evaluate 2D embeddings (mostly umaps)
        )

    results = scgraph.main() # Run the analysis, return a pandas dataframe
    results.to_csv(f'{work_folder}/{slide_id}/scgraph_metrics.csv', index=False)


    # scib metrics ------------------------------------
    adata = ad.read_h5ad(f'{work_folder}/{slide_id}/latents_adata.h5ad')

    bm = Benchmarker(
        adata,
        batch_key=batch_key,
        label_key=labels_key,
        embedding_obsm_keys=["cellina_basal", "cellina_spatial"],  # list your embeddings
        batch_correction_metrics=None,
        bio_conservation_metrics=BioConservation(),
        n_jobs=4, # da povećan ovo? vjerojatno da ali na koliko 16?
    )
    bm.benchmark()
    bm.plot_results_table()
    bm.save_results(f'{work_folder}/{slide_id}/scib_metrics.csv') 
